In [54]:
import torch

print(torch.__version__)
print("PyTorch is working")

2.14.1+cpu
PyTorch is working


In [55]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re

In [56]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 520.96it/s]


In [57]:
tokens = tokenizer.encode('It was good but couldve been better. Great', return_tensors='pt')

In [58]:
result = model(tokens)

In [59]:
result.logits

tensor([[-2.7768, -1.2353,  1.4419,  1.9804,  0.4584]],
       grad_fn=<AddmmBackward0>)

In [60]:
int(torch.argmax(result.logits))+1

4

In [62]:
r = requests.get('https://www.target.com/p/apple-iphone-16-pro/-/A-93597960?preselect=90539855')
soup = BeautifulSoup(r.text, 'html.parser')
results = soup.find_all("div", {"data-test": "review-card--text"})
reviews = [result.get_text(strip=True) for result in results]

In [63]:
print("Status:", r.status_code)
print("HTML length:", len(r.text))
print("Review text exists:", "review-card--text" in r.text)

Status: 200
HTML length: 653857
Review text exists: False


In [ ]:
# text = "Beware of ordering"

# position = r.text.find(text)

# print(position)
# print(r.text[position - 500 : position + 1000])

-1



In [ ]:
# for script in soup.find_all("script"):
#     if "Beware of ordering" in script.get_text():
#         print(script.get_text()[:2000])

In [68]:
import json
import pandas as pd

all_reviews = []

def find_review_texts(obj):
    reviews = []

    if isinstance(obj, dict):
        for key, value in obj.items():

            if key == "text" and isinstance(value, str):
                reviews.append(value)

            else:
                reviews.extend(find_review_texts(value))

    elif isinstance(obj, list):
        for item in obj:
            reviews.extend(find_review_texts(item))

    return reviews


for script in soup.find_all("script"):

    script_text = script.string

    if not script_text:
        continue

    try:
        data = json.loads(script_text)

        reviews_found = find_review_texts(data)

        all_reviews.extend(reviews_found)

    except json.JSONDecodeError:
        continue


# Remove duplicates
all_reviews = list(dict.fromkeys(all_reviews))

# Put into dataframe
df = pd.DataFrame(all_reviews, columns=["review"])

df

,review
0,"7,059"
1,4.4
2,7059 guest ratings
3,Call 855-996-1904 to order now or order online...
4,"Love the phone, so glad I got it. Yeah"
5,It's really good I like the quality of the pho...
6,Phone works well and is inline with expected i...
7,Very good upgrade coming from Iphone 13 pro max
8,Nice phone with good update coming from 13
9,Sound quality is great picture quality is grea...


In [69]:
all_reviews

['7,059',
 '4.4',
 '7059 guest ratings',
 'Call 855-996-1904 to order now or order online – Learn How',
 'Love the phone, so glad I got it. Yeah',
 "It's really good I like the quality of the phone it's easier to use than my other phone overly all 10 out of 10. I love when I take picture on it and when I play games it doesn't lagged and when I watch YouTube videos it's really easy to see and what I love must is when I'm out it still works without Wi-Fi.",
 'Phone works well and is inline with expected iPhone quality!',
 'Very good upgrade coming from Iphone 13 pro max',
 'Nice phone with good update coming from 13',
 'Sound quality is great picture quality is great but touchscreen not very sensitive sometimes it’s very difficult to for example hit the back button or to cancel out windows or web pages most apps freeze on me while I’m using But I’ll always choose IPhone over any other',
 'Great phone but not durable. The first day I had it, I was holding it above a table and it slipped o

In [70]:
import numpy as np
import pandas as pd

df = pd.DataFrame(
    np.array(all_reviews),
    columns=['review']
)

df

,review
0,"7,059"
1,4.4
2,7059 guest ratings
3,Call 855-996-1904 to order now or order online...
4,"Love the phone, so glad I got it. Yeah"
5,It's really good I like the quality of the pho...
6,Phone works well and is inline with expected i...
7,Very good upgrade coming from Iphone 13 pro max
8,Nice phone with good update coming from 13
9,Sound quality is great picture quality is grea...


In [71]:
def sentiment_score(review):
    tokens = tokenizer.encode(review, return_tensors='pt')
    result = model(tokens)
    return int(torch.argmax(result.logits))+1

In [72]:
sentiment_score(df['review'].iloc[11])

5

In [73]:
df['sentiment'] = df['review'].apply(lambda x: sentiment_score(x[:512]))
df

,review,sentiment
0,"7,059",1
1,4.4,4
2,7059 guest ratings,3
3,Call 855-996-1904 to order now or order online...,5
4,"Love the phone, so glad I got it. Yeah",5
5,It's really good I like the quality of the pho...,5
6,Phone works well and is inline with expected i...,5
7,Very good upgrade coming from Iphone 13 pro max,5
8,Nice phone with good update coming from 13,4
9,Sound quality is great picture quality is grea...,3
